# Knowledge distillation: EfficientNetV2S -> MyCNN

## What is knowledge distillation?

Knowledge distillation trains a smaller 'student' model (MyCNN) using
soft probability distributions from a larger 'teacher' model (EfficientNetV2S)
rather than hard one-hot labels.

**Why this helps:**
Hard labels say 'this is Rembrandt (1.0), not Van Gogh (0.0)'.
Soft labels say 'this is probably Rembrandt (0.82), somewhat Van Gogh (0.11),
possibly Caravaggio (0.04)...'. The inter-class similarities encoded in the
teacher's distribution carry structural information about the problem that
hard labels discard entirely.

**The loss function:**

```
L = alpha * T^2 * KL(softmax(teacher/T) || softmax(student/T))
  + (1 - alpha) * CE(student, hard_labels)
```

- T (temperature): higher T softens both distributions, amplifying
  the signal from small probabilities. T=4 is a standard starting point.
- alpha: weight on the distillation term. 0.7 means the student
  primarily learns from the teacher, with hard labels as a regulariser.
- T^2 scaling: compensates for the smaller gradient magnitude that
  results from soft targets (derived in the original Hinton 2015 paper).

## Requirements

- A trained EfficientNetV2S checkpoint (ckpt_phase2_efficientnetv2s.keras)
- The MyCNN class definition (copy from your training notebook)
- The same train/val/test datasets used for MyCNN training


## 0 - Imports and config

In [1]:
import os, math
import numpy as np
from pathlib import Path
import tensorflow as tf
import keras
from keras import Model, layers
from keras.losses import CategoricalCrossentropy, KLDivergence
from keras.metrics import CategoricalAccuracy, AUC
from keras.callbacks import ModelCheckpoint, CSVLogger, EarlyStopping, LearningRateScheduler
import tensorflow_addons as tfa


In [2]:
DATA_DIR    = Path('../wikiart_split')
CKPT_DIR    = Path('./Checkpoints')               # where your teacher checkpoint lives
METRICS_DIR = Path('./Metrics')
KN_DST_DIR  = Path('./knowledge_distillation')
IMAGE_SIZE  = (320, 320)              # MyCNN's resolution
BATCH_SIZE  = 32
N_CLASSES   = 23
AUTOTUNE    = tf.data.AUTOTUNE
seed        = 123

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU: {[g.name for g in gpus]}')


GPU: ['/physical_device:GPU:0']


## 1 - Load MyCNN definition

Paste your current MyCNN and ResidualBlock class definitions in the cell below.

In [3]:
class ResidualBlock(layers.Layer):
    """
    Single residual block: Conv → BN → Activation + shortcut projection.

    Storing conv/bn/activation as named attributes of a Layer subclass
    guarantees Keras tracks their weights correctly.
    The original bug stored these inside plain Python dicts inside a plain
    Python list — Keras never registered them, so they were never trained.
    """

    def __init__(self, in_filters, filters, kernel_size, stride, activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.filters     = filters
        self.kernel_size = kernel_size
        self.stride      = stride
        self.activation  = activation

        # He normal: correct initialisation for ReLU networks
        init = "he_normal"

        self.needs_projection = (stride != 1) or (in_filters != filters)  # <-- computed once, stored

        self.conv1     = layers.Conv2D(filters, kernel_size, strides=stride,
                                      padding="same", use_bias=False,
                                      kernel_initializer=init)
        self.bn1       = layers.BatchNormalization(momentum=0.9)
        self.actv1     = layers.Activation(activation)

        self.conv2     = layers.Conv2D(filters, (3, 3), strides=1,
                                      padding="same", use_bias=False,
                                      kernel_initializer=init)
        self.bn2       = layers.BatchNormalization(momentum=0.9)

        # Only build these layers if they'll actually be used
        if self.needs_projection:
            self.shortcut_conv = layers.Conv2D(filters, (1, 1), strides=stride,
                                               padding="same", use_bias=False,
                                               kernel_initializer=init)
            self.shortcut_bn   = layers.BatchNormalization(momentum=0.9)

        self.add       = layers.Add()
        self.actv2 = layers.Activation(activation)

    def call(self, x, training=False):
        skip = x
        if self.needs_projection:
            skip = self.shortcut_conv(skip)
            skip = self.shortcut_bn(skip, training=training)
    
        x = self.conv1(x)
        x = self.bn1(x, training=training)
        x = self.actv1(x)         # activation between the two convs
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = self.add([x, skip])
        return self.actv2(x)      # activation after the merge

    def get_config(self):
        return {**super().get_config(),
                "filters": self.filters, "kernel_size": self.kernel_size,
                "stride": self.stride,   "activation": self.activation}

class MyCNN(Model):
    def __init__(self, conv_configs, dense_configs, num_classes, augmentation_layer=None, activation="relu", dropout_rate=0.3, **kwargs):
        super().__init__(**kwargs, name="my_cnn")
        self.num_classes = num_classes
        self.conv_configs = conv_configs
        self.dense_configs = dense_configs
        self.augmentation_layer = augmentation_layer
        self.activation = activation
        self.dropout_rate = dropout_rate

        # 1. ADD RESCALING HERE (The fix for your Transfer Learning compatibility)
        self.rescaling = layers.Rescaling(1./255)

        # Store as a Python list of Layer objects assigned to self.
        # Keras DOES track a list of Layers set as an attribute via __setattr__,
        # as long as the list itself is set at attribute assignment time (not grown later).
        # Safest pattern: build the full list first, then assign once.
        blocks_list = []
        in_f = 3  # RGB input — or 1 if grayscale
        for i, (f, k, s) in enumerate(conv_configs):
            blocks_list.append(
                ResidualBlock(in_f, f, k, s, activation=activation, name=f"block_{i}")
            )
            in_f = f  # output of this block becomes input of the next
        self.blocks=blocks_list


        self.gap = layers.GlobalAveragePooling2D(name="GAP")
        dense_list = []
        for i, u in enumerate(self.dense_configs):
            dense_list.append(layers.Dense(u, activation=self.activation, name=f"fc_{i}"))
            dense_list.append(layers.Dropout(self.dropout_rate, name=f"drop_{i}"))
        self.dense_layers = dense_list
        self.classifier = layers.Dense(self.num_classes, activation='softmax', name="head")

    def get_config(self):
        # Obtain the base configuration from the superclass
        config = super().get_config()
        # Add the custom arguments to the dictionary
        config.update({
            "num_classes": self.num_classes,
            "conv_configs": self.conv_configs,
            "dense_configs": self.dense_configs,
            "augmentation_layer": self.augmentation_layer,
            "activation": self.activation,
            "dropout_rate": self.dropout_rate,
        })
        return config

    def call(self, inputs, training=False):
        x = self.rescaling(inputs)
        if self.augmentation_layer is not None:
            x = self.augmentation_layer(x, training=training)
        for block in self.blocks:
            x = block(x, training=training)
        x = self.gap(x)
        for layer in self.dense_layers:
            # Dropout needs training flag; Dense does not
            x = layer(x, training=training) if isinstance(layer, layers.Dropout) else layer(x)
        return self.classifier(x)


print('Paste MyCNN and ResidualBlock definitions above this line, then re-run.')


Paste MyCNN and ResidualBlock definitions above this line, then re-run.


## 2 - Load teacher and datasets

In [ ]:
# Load the best fine-tuned EfficientNetV2S as the teacher.
# This model was trained at IMAGE_SIZE=(384,384) but we will
# use it to generate soft labels for 320x320 images.
# The teacher runs at INFERENCE only - its weights are never updated.
teacher_ckpt = CKPT_DIR / 'ckpt_phase2_transfer_effnetv2s.tf'
teacher      = keras.models.load_model(teacher_ckpt)
teacher.trainable = False
print(f'Teacher loaded: {teacher_ckpt}')

# Separate teacher dataset at 384x384 (teacher's native resolution)
TEACHER_SIZE = (384, 384)
teacher_train_ds = (
    tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / 'train',
        label_mode='categorical',
        batch_size=BATCH_SIZE,
        image_size=TEACHER_SIZE,
        crop_to_aspect_ratio=True,
        shuffle=False,   # CRITICAL: must stay in same order as student dataset
        seed=seed,
    )
    .cache().prefetch(AUTOTUNE)
)


Teacher loaded: Checkpoints\ckpt_phase2_transfer_effnetv2s.tf
Found 9326 files belonging to 23 classes.


In [5]:
# Generate and cache soft labels from the teacher.
# This is done once before training begins.
soft_label_path = KN_DST_DIR / 'teacher_soft_labels.npy'

if soft_label_path.exists():
    print('Loading cached soft labels...')
    soft_labels_train = np.load(soft_label_path)
else:
    if not os.path.exists(KN_DST_DIR):
        os.makedirs(KN_DST_DIR)
    print('Generating soft labels from teacher (one pass over training set)...')
    soft_labels_train = teacher.predict(teacher_train_ds, verbose=1)
    np.save(soft_label_path, soft_labels_train)
    print(f'Saved to {soft_label_path}  shape={soft_labels_train.shape}')

print(f'Soft labels shape: {soft_labels_train.shape}')
print(f'Sample (first image): {soft_labels_train[0][:5]}...')
print(f'Max confidence: {soft_labels_train.max(axis=1).mean():.4f} (avg over training set)')


Loading cached soft labels...
Soft labels shape: (9326, 23)
Sample (first image): [0.31325132 0.01010917 0.00196383 0.0250645  0.00509425]...
Max confidence: 0.7652 (avg over training set)


In [6]:
# Student datasets at 224x224 (MyCNN's resolution).
# shuffle=False on train_raw so image order matches soft_labels_train.
train_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'train',
    label_mode='categorical',
    batch_size=None,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,   # order must match soft labels above
    seed=seed,
)
val_ds = (
    tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / 'val',
        label_mode='categorical',
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE,
        crop_to_aspect_ratio=True,
        shuffle=False,
        seed=seed,
    )
    .cache().prefetch(AUTOTUNE)
)
test_ds = (
    tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / 'test',
        label_mode='categorical',
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE,
        crop_to_aspect_ratio=True,
        shuffle=False,
        seed=seed,
    )
    .cache().prefetch(AUTOTUNE)
)

class_names = train_raw.class_names
print(f'Classes ({len(class_names)}): {class_names}')


Found 9326 files belonging to 23 classes.
Found 1992 files belonging to 23 classes.
Found 2022 files belonging to 23 classes.
Classes (23): ['Albrecht_Durer', 'Boris_Kustodiev', 'Camille_Pissarro', 'Childe_Hassam', 'Claude_Monet', 'Edgar_Degas', 'Eugene_Boudin', 'Gustave_Dore', 'Ilya_Repin', 'Ivan_Aivazovsky', 'Ivan_Shishkin', 'John_Singer_Sargent', 'Marc_Chagall', 'Martiros_Saryan', 'Nicholas_Roerich', 'Pablo_Picasso', 'Paul_Cezanne', 'Pierre_Auguste_Renoir', 'Pyotr_Konchalovsky', 'Raphael_Kirchner', 'Rembrandt', 'Salvador_Dali', 'Vincent_van_Gogh']


In [7]:
# Build the combined (image, hard_label, soft_label) dataset.
#
# The original approach loaded all images into a numpy array (~5 GB),
# causing a MemoryError. The fix: keep images on disk and load them lazily
# via tf.data, zipping the soft_labels array (tiny, ~0.8 MB) onto the pipeline.
#
# Critical requirement: both datasets must iterate in the SAME ORDER.
# train_raw was loaded with shuffle=False, so its order is deterministic.
# soft_labels_train was generated from teacher_train_ds which also used
# shuffle=False — so the ordering is guaranteed to match.

# Soft labels fit in RAM easily (~9326 x 23 x 4 bytes = 860 KB)
soft_labels_ds = tf.data.Dataset.from_tensor_slices(
    soft_labels_train.astype('float32')
)

# train_raw yields (image, hard_label) pairs in fixed order
# Zip adds soft_label as a third element: ((image, hard_label), soft_label)
# Then restructure to (image, hard_label, soft_label)
train_distill_ds = (
    tf.data.Dataset.zip((train_raw, soft_labels_ds))
    .map(
        lambda img_lbl, soft: (img_lbl[0], img_lbl[1], soft),
        num_parallel_calls=AUTOTUNE
    )
    .shuffle(len(soft_labels_train), seed=seed, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(AUTOTUNE)
)

# Verify the first batch has the right shapes before starting training
for imgs, hard, soft in train_distill_ds.take(1):
    print(f'Images:      {imgs.shape}   dtype={imgs.dtype}')
    print(f'Hard labels: {hard.shape}  dtype={hard.dtype}')
    print(f'Soft labels: {soft.shape}  dtype={soft.dtype}')
    print(f'Sample soft label (first image): {soft[0].numpy()[:5]}...')
    print(f'Hard label sum check: {hard[0].numpy().sum():.1f} (should be 1.0)')
    print(f'Soft label sum check: {soft[0].numpy().sum():.4f} (should be ~1.0)')


Images:      (32, 320, 320, 3)   dtype=<dtype: 'float32'>
Hard labels: (32, 23)  dtype=<dtype: 'float32'>
Soft labels: (32, 23)  dtype=<dtype: 'float32'>
Sample soft label (first image): [0.00107845 0.00080751 0.00083031 0.00030027 0.0012644 ]...
Hard label sum check: 1.0 (should be 1.0)
Soft label sum check: 1.0000 (should be ~1.0)


## 3 - Distillation training loop

We use a custom training loop rather than model.fit() because the
distillation loss requires both the hard labels and the teacher's soft
labels, and Keras's standard fit() API only supports a single target tensor.


In [8]:
def distillation_loss(y_hard, y_soft_teacher, y_pred_logits,
                      temperature=4.0, alpha=0.7):
    """
    Combined distillation + standard cross-entropy loss.

    Args:
        y_hard:          one-hot hard labels, shape (B, C)
        y_soft_teacher:  teacher softmax probabilities, shape (B, C)
        y_pred_logits:   student logits (pre-softmax), shape (B, C)
        temperature:     T - higher values soften both distributions
        alpha:           weight on distillation term

    Returns:
        scalar loss value
    """
    T = temperature

    # Soft targets: teacher probabilities raised to 1/T then renormalised.
    # Taking log of teacher probs then dividing by T is equivalent.
    log_soft_teacher = tf.math.log(tf.clip_by_value(y_soft_teacher, 1e-8, 1.0)) / T
    soft_teacher     = tf.nn.softmax(log_soft_teacher)

    # Student soft predictions at temperature T
    soft_student = tf.nn.softmax(y_pred_logits / T)

    # KL divergence: KL(teacher || student)
    # = sum(teacher * log(teacher/student))
    kl = tf.reduce_mean(
        tf.reduce_sum(
            soft_teacher * (tf.math.log(soft_teacher + 1e-8) - tf.math.log(soft_student + 1e-8)),
            axis=1
        )
    )

    # Standard CE with hard labels (label smoothing 0.1)
    ce = tf.reduce_mean(
        tf.keras.losses.categorical_crossentropy(
            y_hard, tf.nn.softmax(y_pred_logits), label_smoothing=0.1
        )
    )

    # T^2 scaling on KL term: compensates for the 1/T^2 reduction in
    # gradient magnitude from soft targets (Hinton et al. 2015)
    return alpha * (T ** 2) * kl + (1.0 - alpha) * ce


In [9]:
def cosine_warmup_fn(base_lr, total_epochs, warmup_epochs=5):
    def fn(epoch):
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        p = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1.0 + math.cos(math.pi * p))
    return fn


def make_train_step(student, optimizer, temperature, alpha):
    """
    Returns a @tf.function-compiled training step.

    Defining the step as a separate function and decorating with @tf.function
    compiles it into a TF graph once on the first call, then runs as a fast
    GPU kernel for all subsequent batches.

    Why not just decorate inside the loop:
    - The augmentation layers inside MyCNN.call() use ops with no XLA converter.
      Removing augmentation_layer from the student (see Cell 15) eliminates the
      while_loop warnings. The compiled step then runs cleanly.
    - loss.numpy() inside a loop forces CPU-GPU sync after every batch.
      Returning tensors and calling .numpy() once per epoch avoids this.
    """
    T     = float(temperature)
    alpha = float(alpha)

    @tf.function
    def train_step(images, hard_labels, soft_labels):
        with tf.GradientTape() as tape:
            probs  = student(images, training=True)
            logits = tf.math.log(tf.clip_by_value(probs, 1e-8, 1.0))

            # Distillation: soften both distributions with temperature T
            log_soft_t   = tf.math.log(tf.clip_by_value(soft_labels, 1e-8, 1.0)) / T
            soft_teacher = tf.nn.softmax(log_soft_t)
            soft_student = tf.nn.softmax(logits / T)
            kl = tf.reduce_mean(tf.reduce_sum(
                soft_teacher * (tf.math.log(soft_teacher + 1e-8)
                                - tf.math.log(soft_student + 1e-8)),
                axis=1
            ))

            # Standard CE with hard labels + label smoothing
            ce = tf.reduce_mean(
                tf.keras.losses.categorical_crossentropy(
                    hard_labels, probs, label_smoothing=0.1
                )
            )

            loss = alpha * (T ** 2) * kl + (1.0 - alpha) * ce

        grads = tape.gradient(loss, student.trainable_variables)
        grads = [tf.clip_by_norm(g, 1.0) if g is not None else g for g in grads]
        optimizer.apply_gradients(zip(grads, student.trainable_variables))
        return loss, probs

    return train_step


@tf.function
def val_step(student, images, hard_labels):
    probs  = student(images, training=False)
    v_loss = tf.reduce_mean(
        tf.keras.losses.categorical_crossentropy(hard_labels, probs)
    )
    return v_loss, probs


def train_with_distillation(
    student, train_ds, val_ds,
    epochs=64, base_lr=1e-3, weight_decay=1e-4,
    temperature=4.0, alpha=0.7,
    ckpt_path='ckpt_mycnn_distilled.keras',
    log_path='log_mycnn_distilled.csv',
):
    optimizer   = tfa.optimizers.AdamW(learning_rate=base_lr, weight_decay=weight_decay)
    lr_schedule = cosine_warmup_fn(base_lr, epochs, warmup_epochs=5)

    # Compile the step once here — subsequent calls reuse the graph
    train_step = make_train_step(student, optimizer, temperature, alpha)

    acc_metric  = CategoricalAccuracy(name='accuracy')
    val_acc_m   = CategoricalAccuracy(name='val_accuracy')
    f1_metric   = tfa.metrics.F1Score(num_classes=N_CLASSES, average='macro', name='f1')
    val_f1_m    = tfa.metrics.F1Score(num_classes=N_CLASSES, average='macro', name='val_f1')

    best_val_loss = float('inf')
    patience_ctr  = 0
    patience      = 10
    best_weights  = None
    log_rows      = []

    for epoch in range(epochs):
        new_lr = lr_schedule(epoch)
        optimizer.learning_rate.assign(new_lr)

        # ── Training ────────────────────────────────────────────────────────
        acc_metric.reset_state(); f1_metric.reset_state()
        # Accumulate loss as a Python float — sum tensors, call .numpy() once
        epoch_loss_sum = 0.0
        n_batches      = 0

        for images, hard_labels, soft_labels in train_ds:
            loss, probs = train_step(images, hard_labels, soft_labels)
            epoch_loss_sum += loss.numpy()   # one sync per batch, unavoidable
            acc_metric.update_state(hard_labels, probs)
            f1_metric.update_state(hard_labels, probs)
            n_batches += 1

        train_loss = epoch_loss_sum / n_batches
        train_acc  = acc_metric.result().numpy()
        train_f1   = f1_metric.result().numpy()

        # ── Validation ──────────────────────────────────────────────────────
        val_acc_m.reset_state(); val_f1_m.reset_state()
        val_loss_sum = 0.0
        val_batches  = 0

        for images, hard_labels in val_ds:
            v_loss, probs = val_step(student, images, hard_labels)
            val_loss_sum += v_loss.numpy()
            val_acc_m.update_state(hard_labels, probs)
            val_f1_m.update_state(hard_labels, probs)
            val_batches += 1

        val_loss = val_loss_sum / val_batches
        val_acc  = val_acc_m.result().numpy()
        val_f1   = val_f1_m.result().numpy()

        print(f'Epoch {epoch+1:03d}/{epochs}  '
              f'loss={train_loss:.4f}  acc={train_acc:.4f}  f1={train_f1:.4f}  '
              f'val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  val_f1={val_f1:.4f}  '
              f'lr={new_lr:.2e}')

        log_rows.append({
            'epoch': epoch, 'loss': train_loss, 'accuracy': train_acc, 'f1': train_f1,
            'val_loss': val_loss, 'val_accuracy': val_acc, 'val_f1': val_f1, 'lr': new_lr,
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights  = student.get_weights()
            student.save(ckpt_path)
            patience_ctr  = 0
            print(f'  Checkpoint saved (best val_loss: {best_val_loss:.4f})')
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    if best_weights is not None:
        student.set_weights(best_weights)

    import csv
    with open(log_path, 'w', newline='') as csvf:
        writer = csv.DictWriter(csvf, fieldnames=log_rows[0].keys())
        writer.writeheader()
        writer.writerows(log_rows)

    print(f'Training complete. Best val_loss: {best_val_loss:.4f}')
    return student


## 4 - Instantiate and train the student

In [10]:
conv_setup = [
    (64,  (7, 7), 2),
    (64,  (3, 3), 1),
    (128, (3, 3), 2),
    (128, (3, 3), 1),
    (256, (3, 3), 2),
    (256, (3, 3), 1),
    (512, (3, 3), 2),
    (512, (3, 3), 1),
]
dense_setup = [512, 256]

# Augmentation is intentionally NOT passed to the student here.
#
# Reason 1 — performance: Keras augmentation ops (RandomRotation,
# RandomTranslation, etc.) have no XLA/graph converter. When @tf.function
# traces the model they fall back to while_loop, generating hundreds of
# warnings and preventing proper graph compilation.
#
# Reason 2 — redundancy: the teacher's soft labels already carry rich
# regularisation signal. The inter-class probability distribution encodes
# structural similarity between artists that hard labels discard. Adding
# augmentation on top provides diminishing returns during distillation.
#
# If you want augmentation, apply it on the dataset pipeline (before
# the model sees the images) rather than inside call() — that way it
# runs as a data-loading op outside the @tf.function-compiled train step.
student = MyCNN(
    augmentation_layer=None,
    conv_configs=conv_setup,
    dense_configs=dense_setup,
    num_classes=N_CLASSES,
    dropout_rate=0.3,
)

student(tf.zeros((1, *IMAGE_SIZE, 3)), training=False)
trainable = sum(np.prod(v.shape) for v in student.trainable_variables)
print(f'Student MyCNN: {trainable:,} trainable parameters')


Student MyCNN: 11,539,735 trainable parameters


In [11]:
student_distilled = train_with_distillation(
    student,
    train_distill_ds,
    val_ds,
    epochs=64,
    base_lr=1e-3,
    weight_decay=1e-4,
    temperature=4.0,    # softens teacher distribution to amplify inter-class signals
    alpha=0.7,          # 70% distillation loss, 30% standard CE
    ckpt_path=CKPT_DIR / 'ckpt_mycnn_distilled.tf',
    log_path=METRICS_DIR / 'log_mycnn_distilled.csv',
)


Epoch 001/64  loss=1.8720  acc=0.2172  f1=0.1387  val_loss=2.5795  val_acc=0.2555  val_f1=0.1555  lr=2.00e-04


INFO:tensorflow:Assets written to: Checkpoints\ckpt_mycnn_distilled.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_mycnn_distilled.tf\assets


  Checkpoint saved (best val_loss: 2.5795)
Epoch 002/64  loss=1.7495  acc=0.2676  f1=0.1846  val_loss=2.3842  val_acc=0.2907  val_f1=0.2178  lr=4.00e-04


INFO:tensorflow:Assets written to: Checkpoints\ckpt_mycnn_distilled.tf\assets


...

INFO:tensorflow:Assets written to: Checkpoints\ckpt_mycnn_distilled.tf\assets


  Checkpoint saved (best val_loss: 1.1648)
Epoch 051/64  loss=0.4782  acc=0.9925  f1=0.9926  val_loss=1.1774  val_acc=0.6883  val_f1=0.6552  lr=1.33e-04
Epoch 052/64  loss=0.4770  acc=0.9909  f1=0.9914  val_loss=1.1763  val_acc=0.6737  val_f1=0.6412  lr=1.15e-04
Epoch 053/64  loss=0.4723  acc=0.9945  f1=0.9948  val_loss=1.2031  val_acc=0.6682  val_f1=0.6255  lr=9.86e-05
Epoch 054/64  loss=0.4704  acc=0.9940  f1=0.9942  val_loss=1.1705  val_acc=0.6792  val_f1=0.6470  lr=8.33e-05
Epoch 055/64  loss=0.4687  acc=0.9948  f1=0.9947  val_loss=1.1634  val_acc=0.6878  val_f1=0.6538  lr=6.92e-05


INFO:tensorflow:Assets written to: Checkpoints\ckpt_mycnn_distilled.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_mycnn_distilled.tf\assets


  Checkpoint saved (best val_loss: 1.1634)
Epoch 056/64  loss=0.4682  acc=0.9954  f1=0.9954  val_loss=1.1687  val_acc=0.6963  val_f1=0.6661  lr=5.63e-05
Epoch 057/64  loss=0.4710  acc=0.9958  f1=0.9958  val_loss=1.1736  val_acc=0.6867  val_f1=0.6539  lr=4.47e-05
Epoch 058/64  loss=0.4685  acc=0.9959  f1=0.9961  val_loss=1.1747  val_acc=0.6862  val_f1=0.6489  lr=3.43e-05
Epoch 059/64  loss=0.4695  acc=0.9955  f1=0.9959  val_loss=1.1687  val_acc=0.6883  val_f1=0.6551  lr=2.53e-05
Epoch 060/64  loss=0.4743  acc=0.9951  f1=0.9949  val_loss=1.1857  val_acc=0.6903  val_f1=0.6536  lr=1.76e-05
Epoch 061/64  loss=0.4785  acc=0.9952  f1=0.9952  val_loss=1.1997  val_acc=0.6832  val_f1=0.6484  lr=1.13e-05
Epoch 062/64  loss=0.4873  acc=0.9947  f1=0.9949  val_loss=1.2160  val_acc=0.6878  val_f1=0.6538  lr=6.37e-06
Epoch 063/64  loss=0.5058  acc=0.9929  f1=0.9929  val_loss=1.2567  val_acc=0.6867  val_f1=0.6510  lr=2.83e-06
Epoch 064/64  loss=0.5383  acc=0.9926  f1=0.9933  val_loss=1.3360  val_acc=0.

## 5 - Evaluate and compare with baseline MyCNN

In [12]:
# Compile the model - the training was custom so this was skipped
student_distilled.compile(
    optimizer=tfa.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss=CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        CategoricalAccuracy(name='accuracy'),
        AUC(multi_label=True, name='auc'),
        tfa.metrics.F1Score(num_classes=N_CLASSES, average='macro', name='f1_score'),
    ]
)

In [13]:
# Distilled model
distilled_results = student_distilled.evaluate(test_ds, return_dict=True, verbose=0)
print('Distilled MyCNN - test results:')
for k, v in distilled_results.items():
    print(f'  {k}: {v:.4f}')

# Baseline MyCNN (load your best non-distilled checkpoint for comparison)
baseline_ckpt = CKPT_DIR / 'checkpoint_my_cnn'
if baseline_ckpt.exists():
    baseline = keras.models.load_model(baseline_ckpt)
    baseline_results = baseline.evaluate(test_ds, return_dict=True, verbose=0)

    print('=' * 52)
    print('Comparison: baseline MyCNN vs distilled MyCNN')
    print('=' * 52)
    print(f"{'Metric':<15} {'Baseline':>10} {'Distilled':>11} {'Delta':>8}")
    print('-' * 48)
    for k in distilled_results:
        delta = distilled_results[k] - baseline_results[k]
        sign  = '+' if delta >= 0 else ''
        print(f'{k:<15} {baseline_results[k]:>10.4f} {distilled_results[k]:>11.4f} {sign}{delta:>7.4f}')
else:
    print(f'Baseline checkpoint not found at {baseline_ckpt}')
    print('Run your standard MyCNN training notebook first, then re-run this cell.')


Distilled MyCNN - test results:
  loss: 1.4834
  accuracy: 0.6864
  auc: 0.9632
  f1_score: 0.6564
Comparison: baseline MyCNN vs distilled MyCNN
Metric            Baseline   Distilled    Delta
------------------------------------------------
loss                2.0363      1.4834 -0.5529
accuracy            0.4698      0.6864 + 0.2166
auc                 0.9191      0.9632 + 0.0441
f1_score            0.4327      0.6564 + 0.2237


## Notes on tuning temperature and alpha

If the distilled model does not outperform the baseline, try:

**Temperature (default: 4.0)**
- Too low (T=1): soft labels are nearly as sharp as hard labels; little benefit
- Too high (T=10+): distributions become too uniform; the class-similarity
  signal is washed out
- Sweet spot for most tasks: T=3-6

**Alpha (default: 0.7)**
- Higher alpha (0.9): student learns almost entirely from teacher;
  useful if the teacher is very accurate (>85% F1)
- Lower alpha (0.4): more weight on hard labels; useful if your dataset
  is small and the teacher might be overconfident on some classes

**Teacher quality matters.** The soft labels are only as good as the teacher.
Use your best fine-tuned EfficientNetV2S checkpoint (Phase 2), not Phase 1.
